In [ ]:
print("Databricks compute is ready")

In [ ]:
spark.version

In [ ]:
data = [
    ("Olist", "AWS Big Data Project"),
    ("Bronze", "Raw Data Layer"),
    ("Silver", "Cleaned Data Layer"),
    ("Gold", "Business Analytics Layer")
]

df = spark.createDataFrame(data, ["layer", "description"])

display(df)

STEP 6.1 - PROJECT CONFIGURATION

In [ ]:
PROJECT_NAME = "olist-aws-batch-data-platform-adhi-2026"

S3_BASE_PATH = "s3://olist-aws-batch-data-platform-adhi-2026"

RAW_PATH = f"{S3_BASE_PATH}/raw/"
BRONZE_PATH = f"{S3_BASE_PATH}/bronze/"
SILVER_PATH = f"{S3_BASE_PATH}/silver/"
GOLD_PATH = f"{S3_BASE_PATH}/gold/"

print("Project Configuration")
print("=" * 60)
print(f"Project : {PROJECT_NAME}")
print(f"Raw     : {RAW_PATH}")
print(f"Bronze  : {BRONZE_PATH}")
print(f"Silver  : {SILVER_PATH}")
print(f"Gold    : {GOLD_PATH}")

STEP 6.2 - VALIDATE DATA LAKE FOLDER STRUCTURE

In [ ]:
lake_layers = {
    "raw": RAW_PATH,
    "bronze": BRONZE_PATH,
    "silver": SILVER_PATH,
    "gold": GOLD_PATH
}

for layer, path in lake_layers.items():
    print(f"\n{'=' * 60}")
    print(f"Checking {layer.upper()} layer")
    print(f"Path: {path}")
    
    try:
        files = dbutils.fs.ls(path)
        
        print(f"Status: AVAILABLE")
        print(f"Items found: {len(files)}")
        
        for file in files[:20]:
            print(f"  - {file.name}")
            
    except Exception as e:
        print(f"Status: ERROR")
        print(f"Details: {str(e)}")

In [ ]:
# ============================================================
# STEP 6.2A - PROVISION MEDALLION STORAGE LAYERS
# ============================================================

lake_paths = {
    "raw": RAW_PATH,
    "bronze": BRONZE_PATH,
    "silver": SILVER_PATH,
    "gold": GOLD_PATH
}

for layer, path in lake_paths.items():
    try:
        dbutils.fs.mkdirs(path)
        print(f"SUCCESS | {layer.upper():<6} | {path}")
    except Exception as e:
        print(f"FAILED  | {layer.upper():<6} | {path}")
        print(f"ERROR   | {str(e)}")

In [ ]:
# ============================================================
# STEP 6.2B - REVALIDATE MEDALLION STORAGE STRUCTURE
# ============================================================

for layer, path in lake_paths.items():
    print("=" * 70)
    print(f"Checking {layer.upper()} layer")
    print(f"Path: {path}")

    try:
        items = dbutils.fs.ls(path)

        print("Status: AVAILABLE")
        print(f"Items found: {len(items)}")

        if items:
            for item in items[:20]:
                print(f"  - {item.name}")
        else:
            print("  - Empty (ready for pipeline output)")

    except Exception as e:
        print("Status: ERROR")
        print(f"Details: {str(e)}")

##6.3 Discover the actual Olist datasets

In [ ]:
# ============================================================
# STEP 6.3 - RAW DATASET INVENTORY
# ============================================================

raw_files = dbutils.fs.ls(RAW_PATH)

print(f"Total files/directories found: {len(raw_files)}\n")

for index, file in enumerate(raw_files, start=1):
    print(f"{index}. {file.name}")
    print(f"   Path : {file.path}")
    print(f"   Size : {file.size} bytes")
    print("-" * 60)

##6.4 Create a dataset inventory

In [ ]:
# ============================================================
# STEP 6.4 - CREATE RAW DATASET INVENTORY
# ============================================================

from pyspark.sql import Row

inventory_data = []

for file in raw_files:
    inventory_data.append(
        Row(
            file_name=file.name,
            file_path=file.path,
            file_size_bytes=file.size,
            source_layer="raw"
        )
    )

inventory_df = spark.createDataFrame(inventory_data)

display(inventory_df)

##6.5 Identify file types

In [ ]:
# ============================================================
# STEP 6.5 - CLASSIFY SOURCE FILES
# ============================================================

from pyspark.sql.functions import (
    col,
    lower,
    regexp_extract
)

classified_inventory_df = (
    inventory_df
    .withColumn(
        "file_type",
        regexp_extract(
            lower(col("file_name")),
            r"\.([a-z0-9]+)$",
            1
        )
    )
)

display(
    classified_inventory_df
    .select(
        "file_name",
        "file_type",
        "file_size_bytes",
        "file_path"
    )
)

##Step 6.6 — Define Source Dataset Configuration

In [ ]:
# ============================================================
# STEP 6.6 - AUTO-DISCOVER SOURCE DATASET CONFIGURATION
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType
)
from pyspark.sql import Row

# Auto-discover datasets from S3 structure
source_datasets = []

for item in raw_files:
    # Check if it's a directory (dataset folder)
    if item.name.endswith("/"):
        dataset_name = item.name.rstrip("/")
        
        # List files in this dataset directory
        try:
            dataset_files = dbutils.fs.ls(item.path)
            
            # Find CSV files in the directory
            for file in dataset_files:
                if file.name.endswith(".csv"):
                    source_file = file.name
                    target_table = dataset_name
                    is_active = True
                    
                    source_datasets.append(
                        (dataset_name, source_file, target_table, is_active)
                    )
                    print(f"Discovered: {dataset_name} -> {source_file}")
        except Exception as e:
            print(f"Skipped {dataset_name}: {str(e)}")

if not source_datasets:
    raise Exception(
        "No datasets auto-discovered. Verify RAW_PATH structure: "
        f"{RAW_PATH} should contain subdirectories with CSV files."
    )

dataset_config_schema = StructType([
    StructField("dataset_name", StringType(), False),
    StructField("source_file", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("is_active", BooleanType(), False)
])

dataset_config_df = spark.createDataFrame(
    source_datasets,
    dataset_config_schema
)

print(f"\nTotal datasets discovered: {len(source_datasets)}")

display(
    dataset_config_df.orderBy("dataset_name")
)

##Step 6.7 — Validate Actual Source Files

In [ ]:
# ============================================================
# STEP 6.7 - VALIDATE CONFIGURED SOURCE FILES
# ============================================================

from pyspark.sql.functions import (
    col,
    concat,
    lit,
    when
)

RAW_BASE_PATH = "s3://olist-aws-batch-data-platform-adhi-2026/raw"

configured_files_df = (
    dataset_config_df
    .filter(col("is_active") == True)
    .withColumn(
        "source_path",
        concat(
            lit(RAW_BASE_PATH + "/"),
            col("dataset_name"),
            lit("/"),
            col("source_file")
        )
    )
)

display(
    configured_files_df.select(
        "dataset_name",
        "source_file",
        "source_path",
        "target_table",
        "is_active"
    ).orderBy("dataset_name")
)

##Step 6.8: Actual S3 CSV existence/readability validation

In [ ]:
# ============================================================
# STEP 6.8 - PHYSICAL SOURCE FILE VALIDATION
# ============================================================

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)

validation_results = []

configured_datasets = (
    configured_files_df
    .select("dataset_name", "source_file", "source_path")
    .orderBy("dataset_name")
    .collect()
)

for dataset in configured_datasets:
    try:
        df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "false")
            .csv(dataset["source_path"])
        )

        validation_results.append(
            Row(
                dataset_name=dataset["dataset_name"],
                source_file=dataset["source_file"],
                source_path=dataset["source_path"],
                validation_status="VALID",
                column_count=len(df.columns),
                record_count=df.count(),
                error_message=None
            )
        )

    except Exception as e:
        validation_results.append(
            Row(
                dataset_name=dataset["dataset_name"],
                source_file=dataset["source_file"],
                source_path=dataset["source_path"],
                validation_status="INVALID",
                column_count=None,
                record_count=None,
                error_message=str(e)[:500]
            )
        )

validation_schema = StructType([
    StructField("dataset_name", StringType(), False),
    StructField("source_file", StringType(), False),
    StructField("source_path", StringType(), False),
    StructField("validation_status", StringType(), False),
    StructField("column_count", LongType(), True),
    StructField("record_count", LongType(), True),
    StructField("error_message", StringType(), True)
])

source_validation_df = spark.createDataFrame(
    validation_results,
    validation_schema
)

display(
    source_validation_df.orderBy("dataset_name")
)

##STEP 6.9 — CREATE INGESTION CONTROL DATAFRAME

In [ ]:
# ============================================================
# STEP 6.9 - CREATE INGESTION CONTROL DATAFRAME
# ============================================================

from pyspark.sql.functions import (
    current_timestamp,
    lit,
    when,
    col,
    concat
)

ingestion_control_df = (
    source_validation_df
    .join(
        dataset_config_df.select(
            "dataset_name",
            "target_table",
            "is_active"
        ),
        on="dataset_name",
        how="inner"
    )
    .withColumn(
        "pipeline_status",
        when(
            (col("validation_status") == "VALID") &
            (col("is_active") == True),
            lit("READY_FOR_INGESTION")
        ).otherwise(lit("BLOCKED"))
    )
    .withColumn(
        "bronze_table",
        when(
            col("validation_status") == "VALID",
            concat(lit("bronze."), col("target_table"))
        ).otherwise(lit(None).cast("string"))
    )
    .withColumn(
        "pipeline_run_timestamp",
        current_timestamp()
    )
    .select(
        "dataset_name",
        "source_file",
        "source_path",
        "target_table",
        "bronze_table",
        "is_active",
        "validation_status",
        "column_count",
        "record_count",
        "pipeline_status",
        "pipeline_run_timestamp",
        "error_message"
    )
)

display(
    ingestion_control_df.orderBy("dataset_name")
)

##Step 6.10 — Bronze Layer Ingestion Design

In [ ]:
# ============================================================
# STEP 6.10A - INITIALIZE BRONZE LAYER
# ============================================================

spark.sql("""
CREATE SCHEMA IF NOT EXISTS bronze
""")

spark.sql("""
SHOW SCHEMAS
""").filter("databaseName = 'bronze'").show()

In [ ]:
# ============================================================
# STEP 6.10B - INGEST VALIDATED DATASETS INTO BRONZE
# ============================================================

from pyspark.sql.functions import (
    current_timestamp,
    lit,
    col
)
from pyspark.sql import Row

# Get only datasets approved by the validation/control layer
datasets_to_ingest = (
    ingestion_control_df
    .filter(col("pipeline_status") == "READY_FOR_INGESTION")
    .select(
        "dataset_name",
        "source_path",
        "target_table"
    )
    .orderBy("dataset_name")
    .collect()
)

bronze_ingestion_results = []

for dataset in datasets_to_ingest:

    dataset_name = dataset["dataset_name"]
    source_path = dataset["source_path"]
    target_table = dataset["target_table"]
    bronze_table = f"bronze.{target_table}"

    try:
        # Read source CSV
        source_df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "false")
            .csv(source_path)
        )

        # Add Bronze ingestion metadata
        bronze_df = (
            source_df
            .withColumn("_ingestion_timestamp", current_timestamp())
            .withColumn("_source_file", col("_metadata.file_path"))
            .withColumn("_dataset_name", lit(dataset_name))
        )

        # Write as Delta table
        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(bronze_table)
        )

        bronze_ingestion_results.append(
            Row(
                dataset_name=dataset_name,
                bronze_table=bronze_table,
                ingestion_status="SUCCESS",
                source_record_count=source_df.count(),
                error_message=None
            )
        )

        print(
            f"SUCCESS | {dataset_name} | "
            f"{bronze_table}"
        )

    except Exception as e:

        bronze_ingestion_results.append(
            Row(
                dataset_name=dataset_name,
                bronze_table=bronze_table,
                ingestion_status="FAILED",
                source_record_count=None,
                error_message=str(e)[:500]
            )
        )

        print(
            f"FAILED | {dataset_name} | "
            f"{str(e)[:200]}"
        )


from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)

bronze_audit_schema = StructType([
    StructField("dataset_name", StringType(), False),
    StructField("bronze_table", StringType(), False),
    StructField("ingestion_status", StringType(), False),
    StructField("source_record_count", LongType(), True),
    StructField("error_message", StringType(), True)
])

bronze_ingestion_audit_df = spark.createDataFrame(
    bronze_ingestion_results,
    bronze_audit_schema
)

display(
    bronze_ingestion_audit_df.orderBy("dataset_name")
)

In [ ]:
# ============================================================
# STEP 6.11 - BRONZE LAYER POST-INGESTION VALIDATION
# ============================================================

from pyspark.sql import Row

bronze_validation_results = []

bronze_datasets = (
    bronze_ingestion_audit_df
    .filter(col("ingestion_status") == "SUCCESS")
    .select(
        "dataset_name",
        "bronze_table",
        "source_record_count"
    )
    .orderBy("dataset_name")
    .collect()
)

for dataset in bronze_datasets:

    dataset_name = dataset["dataset_name"]
    bronze_table = dataset["bronze_table"]
    expected_count = dataset["source_record_count"]

    try:
        # Confirm table exists
        table_exists = spark.catalog.tableExists(bronze_table)

        if not table_exists:
            raise Exception(f"Bronze table does not exist: {bronze_table}")

        # Read Bronze table
        bronze_df = spark.table(bronze_table)

        # Validate record count
        bronze_record_count = bronze_df.count()

        # Validate required ingestion metadata
        required_metadata_columns = [
            "_ingestion_timestamp",
            "_source_file",
            "_dataset_name"
        ]

        missing_columns = [
            column_name
            for column_name in required_metadata_columns
            if column_name not in bronze_df.columns
        ]

        # Determine validation status
        if bronze_record_count != expected_count:

            validation_status = "RECORD_COUNT_MISMATCH"

        elif missing_columns:

            validation_status = "METADATA_MISSING"

        else:

            validation_status = "VALIDATED"

        bronze_validation_results.append(
            Row(
                dataset_name=dataset_name,
                bronze_table=bronze_table,
                expected_record_count=expected_count,
                bronze_record_count=bronze_record_count,
                validation_status=validation_status,
                missing_metadata_columns=(
                    ", ".join(missing_columns)
                    if missing_columns
                    else None
                ),
                error_message=None
            )
        )

        print(
            f"{validation_status} | "
            f"{dataset_name} | "
            f"expected={expected_count} | "
            f"bronze={bronze_record_count}"
        )

    except Exception as e:

        bronze_validation_results.append(
            Row(
                dataset_name=dataset_name,
                bronze_table=bronze_table,
                expected_record_count=expected_count,
                bronze_record_count=None,
                validation_status="FAILED",
                missing_metadata_columns=None,
                error_message=str(e)[:500]
            )
        )

        print(
            f"FAILED | {dataset_name} | {str(e)[:200]}"
        )

In [ ]:
# ============================================================
# STEP 6.11B - CREATE BRONZE VALIDATION REPORT
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)

bronze_validation_schema = StructType([
    StructField("dataset_name", StringType(), False),
    StructField("bronze_table", StringType(), False),
    StructField("expected_record_count", LongType(), True),
    StructField("bronze_record_count", LongType(), True),
    StructField("validation_status", StringType(), False),
    StructField("missing_metadata_columns", StringType(), True),
    StructField("error_message", StringType(), True)
])

bronze_validation_df = spark.createDataFrame(
    bronze_validation_results,
    bronze_validation_schema
)

display(
    bronze_validation_df.orderBy("dataset_name")
)

##Step 6.12 — Persist Bronze Ingestion Audit

In [ ]:
# ============================================================
# STEP 6.12 - PERSIST BRONZE INGESTION AUDIT
# ============================================================

from pyspark.sql.functions import (
    current_timestamp,
    lit
)

# Create operational audit schema/database if required
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS audit
""")

# Combine ingestion results with post-ingestion validation
bronze_audit_final_df = (
    bronze_ingestion_audit_df.alias("ingestion")
    .join(
        bronze_validation_df.alias("validation"),
        on=[
            "dataset_name",
            "bronze_table"
        ],
        how="left"
    )
    .select(
        col("dataset_name"),
        col("bronze_table"),
        col("ingestion_status"),
        col("source_record_count"),
        col("expected_record_count"),
        col("bronze_record_count"),
        col("validation_status"),
        col("missing_metadata_columns"),
        col("validation.error_message"),
        current_timestamp().alias("audit_timestamp")
    )
)

display(
    bronze_audit_final_df.orderBy("dataset_name")
)

In [ ]:
# ============================================================
# STEP 6.12A - BUILD FINAL BRONZE INGESTION AUDIT
# ============================================================

from pyspark.sql.functions import (
    col,
    current_timestamp
)

bronze_audit_final_df = (
    bronze_ingestion_audit_df.alias("i")
    .join(
        bronze_validation_df.alias("v"),
        on=[
            col("i.dataset_name") == col("v.dataset_name"),
            col("i.bronze_table") == col("v.bronze_table")
        ],
        how="left"
    )
    .select(
        col("i.dataset_name").alias("dataset_name"),
        col("i.bronze_table").alias("bronze_table"),
        col("i.ingestion_status").alias("ingestion_status"),
        col("i.source_record_count").alias("source_record_count"),
        col("v.expected_record_count").alias("expected_record_count"),
        col("v.bronze_record_count").alias("bronze_record_count"),
        col("v.validation_status").alias("validation_status"),
        col("v.missing_metadata_columns").alias(
            "missing_metadata_columns"
        ),
        col("i.error_message").alias("ingestion_error_message"),
        col("v.error_message").alias("validation_error_message"),
        current_timestamp().alias("audit_timestamp")
    )
)

display(
    bronze_audit_final_df.orderBy("dataset_name")
)

In [ ]:
# ============================================================
# STEP 6.12B - PERSIST AUDIT HISTORY AS DELTA TABLE
# ============================================================

(
    bronze_audit_final_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("audit.bronze_ingestion_audit")
)

print("SUCCESS | Bronze ingestion audit persisted")

In [ ]:
# ============================================================
# STEP 6.12C - VERIFY PERSISTED BRONZE AUDIT
# ============================================================

display(
    spark.table("audit.bronze_ingestion_audit")
    .orderBy(
        col("audit_timestamp").desc(),
        col("dataset_name")
    )
)

###Phase 7 — Silver Layer Design & Transformation

In [ ]:
# ============================================================
# STEP 7.0 - INITIALIZE SILVER LAYER
# ============================================================

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS silver
""")

spark.sql("""
    SHOW SCHEMAS
""").filter("databaseName = 'silver'").show()

In [ ]:
# ============================================================
# STEP 7.1 - INSPECT BRONZE TABLE SCHEMAS
# ============================================================

bronze_tables = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "product_category_translation",
    "products",
    "sellers"
]

for table_name in bronze_tables:
    
    print("=" * 70)
    print(f"BRONZE TABLE: bronze.{table_name}")
    print("=" * 70)
    
    spark.table(f"bronze.{table_name}").printSchema()
    
    print()

In [ ]:
# ============================================================
# STEP 7.1 - PROFILE BRONZE DATA QUALITY
# ============================================================

from pyspark.sql.functions import (
    col,
    trim,
    when,
    count,
    lit
)

bronze_tables = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "product_category_translation",
    "products",
    "sellers"
]

bronze_quality_results = []

for table_name in bronze_tables:

    bronze_df = spark.table(f"bronze.{table_name}")

    total_records = bronze_df.count()

    print("=" * 80)
    print(f"BRONZE DATA QUALITY PROFILE: {table_name}")
    print(f"Total Records: {total_records}")
    print("=" * 80)

    # Check null and blank values for business columns only
    business_columns = [
        column_name
        for column_name in bronze_df.columns
        if not column_name.startswith("_")
    ]

    null_blank_checks = [
        count(
            when(
                col(column_name).isNull() |
                (trim(col(column_name)) == ""),
                1
            )
        ).alias(column_name)
        for column_name in business_columns
    ]

    quality_df = bronze_df.select(*null_blank_checks)

    display(quality_df)

    print()

##STEP 7.2 — Define Silver Transformation Rules

In [ ]:
# ============================================================
# STEP 7.2 - DEFINE SILVER TRANSFORMATION RULES
# ============================================================

silver_transformation_rules = {
    
    "customers": {
        "trim_strings": True,
        "drop_duplicates": ["customer_id"],
        "cast_columns": {
            "customer_zip_code_prefix": "int"
        },
        "required_columns": [
            "customer_id",
            "customer_unique_id"
        ]
    },

    "geolocation": {
        "trim_strings": True,
        "drop_duplicates": [
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng"
        ],
        "cast_columns": {
            "geolocation_zip_code_prefix": "int",
            "geolocation_lat": "double",
            "geolocation_lng": "double"
        }
    },

    "order_items": {
        "trim_strings": True,
        "drop_duplicates": [
            "order_id",
            "order_item_id"
        ],
        "cast_columns": {
            "order_item_id": "int",
            "shipping_limit_date": "timestamp",
            "price": "double",
            "freight_value": "double"
        },
        "required_columns": [
            "order_id",
            "product_id",
            "seller_id"
        ]
    },

    "order_payments": {
        "trim_strings": True,
        "drop_duplicates": [
            "order_id",
            "payment_sequential"
        ],
        "cast_columns": {
            "payment_sequential": "int",
            "payment_installments": "int",
            "payment_value": "double"
        },
        "required_columns": [
            "order_id",
            "payment_type"
        ]
    },

    "order_reviews": {
        "trim_strings": True,
        "drop_duplicates": [
            "review_id",
            "order_id"
        ],
        "cast_columns": {
            "review_score": "int",
            "review_creation_date": "timestamp",
            "review_answer_timestamp": "timestamp"
        },
        "required_columns": [
            "review_id",
            "order_id",
            "review_score"
        ]
    },

    "orders": {
        "trim_strings": True,
        "drop_duplicates": ["order_id"],
        "cast_columns": {
            "order_purchase_timestamp": "timestamp",
            "order_approved_at": "timestamp",
            "order_delivered_carrier_date": "timestamp",
            "order_delivered_customer_date": "timestamp",
            "order_estimated_delivery_date": "timestamp"
        },
        "required_columns": [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp"
        ]
    },

    "product_category_translation": {
        "trim_strings": True,
        "drop_duplicates": [
            "product_category_name"
        ],
        "required_columns": [
            "product_category_name",
            "product_category_name_english"
        ]
    },

    "products": {
        "trim_strings": True,
        "drop_duplicates": ["product_id"],
        "cast_columns": {
            "product_name_lenght": "int",
            "product_description_lenght": "int",
            "product_photos_qty": "int",
            "product_weight_g": "double",
            "product_length_cm": "double",
            "product_height_cm": "double",
            "product_width_cm": "double"
        },
        "required_columns": [
            "product_id"
        ]
    },

    "sellers": {
        "trim_strings": True,
        "drop_duplicates": ["seller_id"],
        "cast_columns": {
            "seller_zip_code_prefix": "int"
        },
        "required_columns": [
            "seller_id"
        ]
    }
}

print("=" * 70)
print("SILVER TRANSFORMATION RULES CONFIGURED")
print("=" * 70)

for dataset_name, rules in silver_transformation_rules.items():
    print(f"\nDataset: {dataset_name}")
    print(f"Duplicate Key : {rules.get('drop_duplicates')}")
    print(f"Type Casts   : {rules.get('cast_columns', {})}")
    print(f"Required     : {rules.get('required_columns', [])}")

In [ ]:
# ============================================================
# STEP 7.3 - BUILD REUSABLE SILVER TRANSFORMATION FUNCTION
# ============================================================

from pyspark.sql.functions import (
    col,
    trim,
    when,
    lit,
    current_timestamp,
    expr
)


def transform_bronze_to_silver(
    bronze_df,
    dataset_name,
    rules
):
    """
    Apply standardized Silver layer transformations.

    Transformations:
    1. Trim string columns
    2. Cast configured columns to target data types
    3. Remove records with missing required columns (after casting)
    4. Remove duplicates using business keys
    5. Add Silver transformation metadata
    """

    silver_df = bronze_df

    # --------------------------------------------------------
    # 1. TRIM STRING COLUMNS
    # --------------------------------------------------------
    if rules.get("trim_strings", False):

        for field in silver_df.schema.fields:

            if field.dataType.simpleString() == "string":

                column_name = field.name

                # Do not modify metadata columns
                if not column_name.startswith("_"):

                    silver_df = silver_df.withColumn(
                        column_name,
                        trim(col(column_name))
                    )

    # --------------------------------------------------------
    # 2. CAST COLUMNS TO TARGET DATA TYPES (BEFORE FILTERING)
    # --------------------------------------------------------
    # Cast first so that invalid values become NULL,
    # then filter on the casted columns
    cast_columns = rules.get("cast_columns", {})

    for column_name, target_type in cast_columns.items():

        if column_name in silver_df.columns:

            silver_df = silver_df.withColumn(
                column_name,
                expr(f"try_cast({column_name} as {target_type})")
            )

    # --------------------------------------------------------
    # 3. REMOVE RECORDS WITH MISSING REQUIRED VALUES (AFTER CASTING)
    # --------------------------------------------------------
    # Apply after casting so we filter on the cleaned/typed values
    required_columns = rules.get("required_columns", [])

    for column_name in required_columns:

        # Check column type to apply appropriate filter
        field = [f for f in silver_df.schema.fields if f.name == column_name][0]
        
        if field.dataType.simpleString() == "string":
            # For strings, check both NULL and empty
            silver_df = silver_df.filter(
                col(column_name).isNotNull() &
                (trim(col(column_name)) != "")
            )
        else:
            # For non-strings (int, double, timestamp), just check NULL
            silver_df = silver_df.filter(
                col(column_name).isNotNull()
            )

    # --------------------------------------------------------
    # 4. REMOVE DUPLICATES USING BUSINESS KEY
    # --------------------------------------------------------
    duplicate_keys = rules.get("drop_duplicates", [])

    if duplicate_keys:

        silver_df = silver_df.dropDuplicates(
            duplicate_keys
        )

    # --------------------------------------------------------
    # 5. ADD SILVER TRANSFORMATION METADATA
    # --------------------------------------------------------
    silver_df = (
        silver_df
        .withColumn(
            "_silver_transformation_timestamp",
            current_timestamp()
        )
        .withColumn(
            "_silver_dataset_name",
            lit(dataset_name)
        )
    )

    return silver_df


print("=" * 70)
print("REUSABLE SILVER TRANSFORMATION FUNCTION CREATED")
print("=" * 70)
print("Function: transform_bronze_to_silver()")
print()
print("Supported transformations:")
print("1. Trim string columns")
print("2. Cast columns to target data types")
print("3. Filter missing required values (after casting)")
print("4. Remove duplicate business keys")
print("5. Add Silver transformation metadata")

In [ ]:
# ============================================================
# STEP 7.4 - EXECUTE BRONZE TO SILVER TRANSFORMATIONS
# ============================================================

silver_transformation_results = []

for dataset_name, rules in silver_transformation_rules.items():

    print("=" * 80)
    print(f"PROCESSING BRONZE TO SILVER: {dataset_name}")
    print("=" * 80)

    # Read Bronze table
    bronze_df = spark.table(f"bronze.{dataset_name}")

    try:
        # Apply reusable transformation function
        silver_df = transform_bronze_to_silver(
            bronze_df,
            dataset_name,
            rules
        )

        # Force evaluation to catch transformation errors
        _ = silver_df.count()

        # Write transformed data to Silver layer
        (
            silver_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(f"silver.{dataset_name}")
        )

        # Validation counts
        bronze_count = bronze_df.count()
        silver_count = spark.table(f"silver.{dataset_name}").count()

        silver_transformation_results.append({
            "dataset": dataset_name,
            "bronze_records": bronze_count,
            "silver_records": silver_count,
            "records_removed": bronze_count - silver_count
        })

        print(f"Bronze Records  : {bronze_count}")
        print(f"Silver Records  : {silver_count}")
        print(f"Records Removed : {bronze_count - silver_count}")
        print(f"Status          : SUCCESS")

    except Exception as e:
        error_msg = str(e)[:300]
        print(f"TRANSFORMATION ERROR: {error_msg}")
        print(f"Status          : FAILED")

        silver_transformation_results.append({
            "dataset": dataset_name,
            "bronze_records": bronze_df.count(),
            "silver_records": 0,
            "records_removed": 0
        })

    print()

print("=" * 80)
print("ALL BRONZE TO SILVER TRANSFORMATIONS COMPLETED")
print("=" * 80)

##STEP 7.5 — Validate Silver Tables

In [ ]:
# ============================================================
# STEP 7.5 - VALIDATE SILVER DATA QUALITY
# ============================================================

from pyspark.sql.functions import (
    col,
    trim,
    when,
    count
)
from pyspark.sql.types import StringType

silver_validation_results = []

for dataset_name, rules in silver_transformation_rules.items():

    print("=" * 80)
    print(f"SILVER DATA QUALITY VALIDATION: {dataset_name}")
    print("=" * 80)

    silver_df = spark.table(f"silver.{dataset_name}")

    total_records = silver_df.count()

    print(f"Total Silver Records: {total_records}")
    print()

    # --------------------------------------------------------
    # 1. Validate required columns
    # --------------------------------------------------------
    required_columns = rules.get("required_columns", [])

    if required_columns:

        required_checks = []

        for column_name in required_columns:

            column_data_type = silver_df.schema[column_name].dataType

            # String columns: check NULL and blank values
            if isinstance(column_data_type, StringType):
                condition = (
                    col(column_name).isNull() |
                    (trim(col(column_name)) == "")
                )

            # Numeric, timestamp and other columns: check NULL only
            else:
                condition = col(column_name).isNull()

            required_checks.append(
                count(
                    when(condition, 1)
                ).alias(column_name)
            )

        required_validation_df = silver_df.select(*required_checks)

        print("Required Column Null/Blank Validation:")
        display(required_validation_df)

    else:
        print("No required columns configured.")

    # --------------------------------------------------------
    # 2. Validate duplicate business keys
    # --------------------------------------------------------
    duplicate_keys = rules.get("drop_duplicates", [])

    if duplicate_keys:

        duplicate_count = (
            silver_df
            .groupBy(*duplicate_keys)
            .count()
            .filter(col("count") > 1)
            .count()
        )

        print(
            f"Duplicate Business Key Groups Remaining: "
            f"{duplicate_count}"
        )

    else:
        duplicate_count = 0
        print("No duplicate key configured.")

    # --------------------------------------------------------
    # Store validation result
    # --------------------------------------------------------
    silver_validation_results.append({
        "dataset": dataset_name,
        "silver_records": total_records,
        "duplicate_key_groups": duplicate_count
    })

    print()
    print(f"VALIDATION STATUS: COMPLETED - {dataset_name}")
    print()

print("=" * 80)
print("ALL SILVER DATA QUALITY VALIDATIONS COMPLETED")
print("=" * 80)

##STEP 7.6 — Create a Consolidated Silver Validation Summary Table/DataFrame

In [ ]:
# ============================================================
# STEP 7.6 - CREATE CONSOLIDATED SILVER VALIDATION SUMMARY
# ============================================================

from pyspark.sql import Row

# ------------------------------------------------------------
# Create DataFrame from validation results
# ------------------------------------------------------------
silver_validation_summary_df = spark.createDataFrame(
    [Row(**result) for result in silver_validation_results]
)

# ------------------------------------------------------------
# Join transformation results with validation results
# ------------------------------------------------------------
silver_transformation_summary_df = spark.createDataFrame(
    [Row(**result) for result in silver_transformation_results]
)

consolidated_silver_summary_df = (
    silver_transformation_summary_df
    .join(
        silver_validation_summary_df,
        on=["dataset", "silver_records"],
        how="left"
    )
    .select(
        "dataset",
        "bronze_records",
        "silver_records",
        "records_removed",
        "duplicate_key_groups"
    )
    .orderBy("dataset")
)

# ------------------------------------------------------------
# Add validation status
# ------------------------------------------------------------
from pyspark.sql.functions import when, lit

consolidated_silver_summary_df = (
    consolidated_silver_summary_df
    .withColumn(
        "validation_status",
        when(
            col("duplicate_key_groups") == 0,
            lit("PASSED")
        ).otherwise(lit("FAILED"))
    )
)

# ------------------------------------------------------------
# Display consolidated summary
# ------------------------------------------------------------
print("=" * 100)
print("CONSOLIDATED SILVER DATA QUALITY SUMMARY")
print("=" * 100)

display(consolidated_silver_summary_df)

print()
print("=" * 100)
print("SILVER VALIDATION SUMMARY COMPLETED")
print("=" * 100)

##PHASE 8 — GOLD LAYER: BUSINESS TRANSFORMATIONS

In [ ]:
# ============================================================
# STEP 8.1 - INSPECT SILVER LAYER FOR GOLD DATA MODEL
# ============================================================

print("=" * 80)
print("SILVER LAYER TABLES AVAILABLE FOR GOLD TRANSFORMATIONS")
print("=" * 80)

silver_tables = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "product_category_translation",
    "products",
    "sellers"
]

for table_name in silver_tables:

    full_table_name = f"silver.{table_name}"

    print()
    print("-" * 80)
    print(f"TABLE: {full_table_name}")
    print("-" * 80)

    df = spark.table(full_table_name)

    print(f"Record Count: {df.count()}")
    print("Columns:")

    for column_name, data_type in df.dtypes:
        print(f"  - {column_name}: {data_type}")

print()
print("=" * 80)
print("SILVER LAYER INSPECTION COMPLETED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 8.2 - DEFINE GOLD LAYER STAR SCHEMA AND RELATIONSHIPS
# ============================================================

gold_data_model = {

    "dimensions": {

        "dim_customers": {
            "source": ["silver.customers"],
            "business_key": ["customer_id"],
            "grain": "One row per customer_id",
            "columns": [
                "customer_id",
                "customer_unique_id",
                "customer_zip_code_prefix",
                "customer_city",
                "customer_state"
            ]
        },

        "dim_sellers": {
            "source": ["silver.sellers"],
            "business_key": ["seller_id"],
            "grain": "One row per seller_id",
            "columns": [
                "seller_id",
                "seller_zip_code_prefix",
                "seller_city",
                "seller_state"
            ]
        },

        "dim_products": {
            "source": [
                "silver.products",
                "silver.product_category_translation"
            ],
            "business_key": ["product_id"],
            "grain": "One row per product_id",
            "columns": [
                "product_id",
                "product_category_name",
                "product_category_name_english",
                "product_name_length",
                "product_description_length",
                "product_photos_qty",
                "product_weight_g",
                "product_length_cm",
                "product_height_cm",
                "product_width_cm"
            ]
        },

        "dim_date": {
            "source": ["Derived from silver.orders"],
            "business_key": ["date_key"],
            "grain": "One row per calendar date"
        }
    },

    "facts": {

        "fact_orders": {
            "source": ["silver.orders"],
            "business_key": ["order_id"],
            "grain": "One row per order_id"
        },

        "fact_order_items": {
            "source": ["silver.order_items"],
            "business_key": ["order_id", "order_item_id"],
            "grain": "One row per order item"
        },

        "fact_payments": {
            "source": ["silver.order_payments"],
            "business_key": [
                "order_id",
                "payment_sequential"
            ],
            "grain": "One row per payment transaction"
        },

        "fact_reviews": {
            "source": ["silver.order_reviews"],
            "business_key": ["review_id"],
            "grain": "One row per review"
        }
    },

    "relationships": [

        {
            "left": "silver.orders.customer_id",
            "right": "silver.customers.customer_id",
            "relationship": "Many orders to one customer"
        },

        {
            "left": "silver.order_items.order_id",
            "right": "silver.orders.order_id",
            "relationship": "Many order items to one order"
        },

        {
            "left": "silver.order_items.product_id",
            "right": "silver.products.product_id",
            "relationship": "Many order items to one product"
        },

        {
            "left": "silver.order_items.seller_id",
            "right": "silver.sellers.seller_id",
            "relationship": "Many order items to one seller"
        },

        {
            "left": "silver.order_payments.order_id",
            "right": "silver.orders.order_id",
            "relationship": "Many payments to one order"
        },

        {
            "left": "silver.order_reviews.order_id",
            "right": "silver.orders.order_id",
            "relationship": "Reviews associated with orders"
        },

        {
            "left": "silver.products.product_category_name",
            "right": (
                "silver.product_category_translation."
                "product_category_name"
            ),
            "relationship": "Product category translation lookup"
        }
    ]
}

print("=" * 80)
print("GOLD LAYER STAR SCHEMA DEFINED")
print("=" * 80)

print("\nDIMENSION TABLES:")
for table_name, details in gold_data_model["dimensions"].items():
    print(
        f"- {table_name}: "
        f"{details['grain']}"
    )

print("\nFACT TABLES:")
for table_name, details in gold_data_model["facts"].items():
    print(
        f"- {table_name}: "
        f"{details['grain']}"
    )

print("\nTABLE RELATIONSHIPS:")
for relationship in gold_data_model["relationships"]:
    print(
        f"- {relationship['left']} "
        f"--> {relationship['right']}"
    )
    print(f"  {relationship['relationship']}")

print()
print("=" * 80)
print("GOLD DATA MODEL DESIGN COMPLETED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 8.3 - CREATE AND VALIDATE GOLD DATABASE/SCHEMA
# ============================================================

from pyspark.sql.functions import col

# Create Gold schema if it does not already exist
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS gold
""")

print("=" * 80)
print("GOLD DATABASE/SCHEMA CREATION")
print("=" * 80)

# Display available schemas to verify Gold was created
schemas_df = spark.sql("""
    SHOW SCHEMAS
""")

print("Available schemas containing 'gold':")
display(
    schemas_df.filter(
        col("databaseName") == "gold"
    )
)

# Verify schema accessibility
spark.sql("USE gold")

print()
print("Current schema successfully set to: gold")
print()

# Check existing Gold tables
gold_tables_df = spark.sql("""
    SHOW TABLES IN gold
""")

print("Existing Gold Tables:")
display(gold_tables_df)

print()
print("=" * 80)
print("GOLD DATABASE/SCHEMA VALIDATION COMPLETED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 8.4 - CREATE GOLD DIMENSION TABLE: dim_customers
# ============================================================

from pyspark.sql.functions import (
    col,
    current_timestamp
)

print("=" * 80)
print("CREATING GOLD DIMENSION TABLE: dim_customers")
print("=" * 80)

# Read Silver customers table
silver_customers_df = spark.table("silver.customers")

# Select dimension attributes
dim_customers_df = (
    silver_customers_df
    .select(
        col("customer_id"),
        col("customer_unique_id"),
        col("customer_zip_code_prefix"),
        col("customer_city"),
        col("customer_state")
    )
    .dropDuplicates(["customer_id"])
    .withColumn(
        "_gold_transformation_timestamp",
        current_timestamp()
    )
)

# Write Customer Dimension to Gold layer
(
    dim_customers_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_customers")
)

# Validation
dim_customer_count = spark.table("gold.dim_customers").count()

print(f"Gold Dimension Table : gold.dim_customers")
print(f"Record Count         : {dim_customer_count}")
print(f"Status               : SUCCESS")

print()
print("=" * 80)
print("dim_customers CREATION COMPLETED")
print("=" * 80)

# Display sample records
display(
    spark.table("gold.dim_customers")
    .orderBy("customer_id")
    .limit(10)
)

In [ ]:
# ============================================================
# STEP 8.5 - CREATE GOLD DIMENSION TABLE: dim_products
# ============================================================

from pyspark.sql.functions import (
    col,
    current_timestamp
)

print("=" * 80)
print("CREATING GOLD DIMENSION TABLE: dim_products")
print("=" * 80)

# Read Silver source tables
silver_products_df = spark.table("silver.products")
silver_category_translation_df = spark.table(
    "silver.product_category_translation"
)

# Create Product Dimension with English category lookup
dim_products_df = (
    silver_products_df.alias("p")
    .join(
        silver_category_translation_df.alias("t"),
        col("p.product_category_name") ==
        col("t.product_category_name"),
        "left"
    )
    .select(
        col("p.product_id"),
        col("p.product_category_name"),
        col("t.product_category_name_english"),
        col("p.product_name_lenght"),
        col("p.product_description_lenght"),
        col("p.product_photos_qty"),
        col("p.product_weight_g"),
        col("p.product_length_cm"),
        col("p.product_height_cm"),
        col("p.product_width_cm")
    )
    .dropDuplicates(["product_id"])
    .withColumn(
        "_gold_transformation_timestamp",
        current_timestamp()
    )
)

# Write Product Dimension to Gold layer
(
    dim_products_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_products")
)

# Validation
dim_product_count = spark.table("gold.dim_products").count()

print(f"Gold Dimension Table : gold.dim_products")
print(f"Record Count         : {dim_product_count}")
print("Status               : SUCCESS")

print()
print("=" * 80)
print("dim_products CREATION COMPLETED")
print("=" * 80)

# Display sample records
display(
    spark.table("gold.dim_products")
    .orderBy("product_id")
    .limit(10)
)

##STEP 8.6 — Create Gold Seller Dimension Table

In [ ]:
from pyspark.sql.functions import current_timestamp

print("=" * 70)
print("CREATING GOLD DIMENSION TABLE: dim_sellers")
print("=" * 70)

# Read Silver sellers table
silver_sellers_df = spark.table("silver.sellers")

# Select business columns and Gold metadata
dim_sellers_df = (
    silver_sellers_df
    .select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    )
    .dropDuplicates(["seller_id"])
    .withColumn("_gold_transformation_timestamp", current_timestamp())
)

# Write Gold dimension table
(
    dim_sellers_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_sellers")
)

# Validation
seller_count = spark.table("gold.dim_sellers").count()

print(f"Gold Dimension Table : gold.dim_sellers")
print(f"Record Count         : {seller_count}")
print(f"Status               : SUCCESS")

print()
print("=" * 70)
print("dim_sellers CREATION COMPLETED")
print("=" * 70)

display(spark.table("gold.dim_sellers"))

##STEP 8.7 — Create Gold Date Dimension Table

In [ ]:
from pyspark.sql import functions as F

print("=" * 70)
print("CREATING GOLD DIMENSION TABLE: dim_date")
print("=" * 70)

# Collect all relevant dates from Silver orders
order_dates_df = (
    spark.table("silver.orders")
    .select(
        F.to_date("order_purchase_timestamp").alias("date")
    )
    .union(
        spark.table("silver.orders")
        .select(F.to_date("order_approved_at").alias("date"))
    )
    .union(
        spark.table("silver.orders")
        .select(F.to_date("order_delivered_carrier_date").alias("date"))
    )
    .union(
        spark.table("silver.orders")
        .select(F.to_date("order_delivered_customer_date").alias("date"))
    )
    .union(
        spark.table("silver.orders")
        .select(F.to_date("order_estimated_delivery_date").alias("date"))
    )
    .filter(F.col("date").isNotNull())
    .distinct()
)

# Create Date Dimension
dim_date_df = (
    order_dates_df
    .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("week_of_year", F.weekofyear("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
    .withColumn("is_weekend",
                F.when(F.dayofweek("date").isin([1, 7]), True)
                 .otherwise(False))
    .withColumn("_gold_transformation_timestamp", F.current_timestamp())
    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "month_name",
        "week_of_year",
        "day",
        "day_of_week",
        "day_name",
        "is_weekend",
        "_gold_transformation_timestamp"
    )
    .orderBy("date")
)

# Write to Gold
(
    dim_date_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.dim_date")
)

# Validation
date_count = spark.table("gold.dim_date").count()

print(f"Gold Dimension Table : gold.dim_date")
print(f"Record Count         : {date_count}")
print(f"Status               : SUCCESS")

print()
print("=" * 70)
print("dim_date CREATION COMPLETED")
print("=" * 70)

display(spark.table("gold.dim_date").orderBy("date"))

##STEP 8.8 — Create Gold Orders Fact Table

In [ ]:
from pyspark.sql import functions as F

print("=" * 70)
print("CREATING GOLD FACT TABLE: fact_orders")
print("=" * 70)

# Read source and dimension tables
silver_orders_df = spark.table("silver.orders")
dim_date_df = spark.table("gold.dim_date").select(
    F.col("date").alias("dim_date"),
    F.col("date_key").alias("order_purchase_date_key")
)

# Create fact table
fact_orders_df = (
    silver_orders_df
    .withColumn(
        "order_purchase_date",
        F.to_date(F.col("order_purchase_timestamp"))
    )
    .join(
        dim_date_df,
        F.col("order_purchase_date") == F.col("dim_date"),
        "left"
    )
    .select(
        "order_id",
        "customer_id",
        "order_purchase_date_key",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "_gold_transformation_timestamp",
        F.current_timestamp()
    )
)

# Write Gold fact table
(
    fact_orders_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_orders")
)

# Validation
fact_order_count = spark.table("gold.fact_orders").count()

print(f"Gold Fact Table      : gold.fact_orders")
print(f"Record Count         : {fact_order_count}")
print(f"Status               : SUCCESS")

print()
print("=" * 70)
print("fact_orders CREATION COMPLETED")
print("=" * 70)

display(
    spark.table("gold.fact_orders")
    .orderBy("order_purchase_timestamp")
)

##STEP 8.9 — Create Gold Order Items Fact Table

In [ ]:
from pyspark.sql.functions import col, current_timestamp, to_date


print("=" * 70)
print("CREATING GOLD FACT TABLE: fact_order_items")
print("=" * 70)

# ============================================================
# CREATE ORDER ITEMS FACT DATAFRAME
# ============================================================

silver_order_items_df = spark.table("silver.order_items")

fact_order_items_df = (
    silver_order_items_df
    .select(
        col("order_id"),
        col("order_item_id"),
        col("product_id"),
        col("seller_id"),
        col("shipping_limit_date"),
        col("price"),
        col("freight_value")
    )
    .withColumn(
        "shipping_date",
        to_date(col("shipping_limit_date"))
    )
    .withColumn(
        "_gold_transformation_timestamp",
        current_timestamp()
    )
)

# ============================================================
# SAVE AS GOLD DELTA TABLE
# ============================================================

(
    fact_order_items_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_order_items")
)

# ============================================================
# VALIDATION
# ============================================================

record_count = spark.table("gold.fact_order_items").count()

print(f"Gold Fact Table    : gold.fact_order_items")
print(f"Record Count       : {record_count}")
print(f"Status             : SUCCESS")

print("\n" + "=" * 70)
print("fact_order_items CREATION COMPLETED")
print("=" * 70)

display(
    spark.table("gold.fact_order_items").limit(10)
)

##STEP 8.10 — Create Gold Payments Fact Table

In [ ]:
from pyspark.sql.functions import col, current_timestamp

print("=" * 70)
print("CREATING GOLD FACT TABLE: fact_payments")
print("=" * 70)

# ============================================================
# CREATE PAYMENTS FACT DATAFRAME
# ============================================================

silver_order_payments_df = spark.table("silver.order_payments")

fact_payments_df = (
    silver_order_payments_df
    .select(
        col("order_id"),
        col("payment_sequential"),
        col("payment_type"),
        col("payment_installments"),
        col("payment_value")
    )
    .withColumn(
        "_gold_transformation_timestamp",
        current_timestamp()
    )
)

# ============================================================
# SAVE AS GOLD DELTA TABLE
# ============================================================

(
    fact_payments_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_payments")
)

# ============================================================
# VALIDATION
# ============================================================

record_count = spark.table("gold.fact_payments").count()

print(f"Gold Fact Table    : gold.fact_payments")
print(f"Record Count       : {record_count}")
print(f"Status             : SUCCESS")

print("\n" + "=" * 70)
print("fact_payments CREATION COMPLETED")
print("=" * 70)

display(
    spark.table("gold.fact_payments").limit(10)
)

##STEP 8.11 — Create Gold Reviews Fact Table

In [ ]:
# ============================================================
# STEP 8.11 — CREATE GOLD REVIEWS FACT TABLE
# ============================================================

from pyspark.sql.functions import current_timestamp

print("=" * 70)
print("CREATING GOLD FACT TABLE: fact_reviews")
print("=" * 70)

# Read Silver order reviews table
silver_order_reviews_df = spark.table("silver.order_reviews")

# Add Gold transformation timestamp
fact_reviews_df = (
    silver_order_reviews_df
    .withColumn("_gold_transformation_timestamp", current_timestamp())
)

# Save as Gold Delta table
fact_reviews_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.fact_reviews")

# Validation
record_count = spark.table("gold.fact_reviews").count()

print()
print(f"Gold Fact Table    : gold.fact_reviews")
print(f"Record Count       : {record_count}")
print(f"Status             : SUCCESS")

print()
print("=" * 70)
print("fact_reviews CREATION COMPLETED")
print("=" * 70)

# Display sample records
display(spark.table("gold.fact_reviews"))

##STEP 8.12 — Validate All Gold Tables

In [ ]:
# ============================================================
# STEP 8.12 — VALIDATE ALL GOLD TABLES
# ============================================================

from pyspark.sql.functions import col

print("=" * 75)
print("VALIDATING ALL GOLD TABLES")
print("=" * 75)

# Expected Gold tables
gold_tables = [
    "gold.dim_sellers",
    "gold.dim_date",
    "gold.fact_orders",
    "gold.fact_order_items",
    "gold.fact_payments",
    "gold.fact_reviews"
]

validation_results = []

for table_name in gold_tables:
    print(f"\nChecking: {table_name}")

    try:
        # Check table exists and get data
        df = spark.table(table_name)
        record_count = df.count()
        column_count = len(df.columns)

        validation_results.append((
            table_name,
            record_count,
            column_count,
            "SUCCESS"
        ))

        print(f"  Record Count : {record_count}")
        print(f"  Column Count : {column_count}")
        print(f"  Status       : SUCCESS")

    except Exception as e:
        validation_results.append((
            table_name,
            0,
            0,
            "FAILED"
        ))

        print(f"  Status       : FAILED")
        print(f"  Error        : {str(e)[:200]}")

# Create validation summary
validation_df = spark.createDataFrame(
    validation_results,
    ["table_name", "record_count", "column_count", "status"]
)

print("\n" + "=" * 75)
print("GOLD LAYER VALIDATION SUMMARY")
print("=" * 75)

display(validation_df)

# Final status
failed_count = validation_df.filter(col("status") == "FAILED").count()

print("\n" + "=" * 75)

if failed_count == 0:
    print("FINAL STATUS : SUCCESS")
    print("ALL GOLD TABLES VALIDATED SUCCESSFULLY")
else:
    print(f"FINAL STATUS : FAILED")
    print(f"FAILED TABLES: {failed_count}")

print("=" * 75)

##STEP 9 — Delta Lake Features

In [ ]:
# ============================================================
# STEP 9.1 — VERIFY GOLD TABLES ARE DELTA TABLES
# ============================================================

gold_tables = [
    "gold.dim_sellers",
    "gold.dim_date",
    "gold.fact_orders",
    "gold.fact_order_items",
    "gold.fact_payments",
    "gold.fact_reviews"
]

print("=" * 80)
print("DELTA LAKE TABLE FORMAT VALIDATION")
print("=" * 80)

delta_validation_results = []

for table_name in gold_tables:

    print(f"\nChecking: {table_name}")

    try:
        detail_df = spark.sql(
            f"DESCRIBE DETAIL {table_name}"
        )

        table_format = detail_df.select("format").first()["format"]
        record_count = spark.table(table_name).count()

        print(f"Format       : {table_format}")
        print(f"Record Count : {record_count}")

        if table_format.lower() == "delta":
            status = "SUCCESS"
            print("Status       : SUCCESS - Delta table")
        else:
            status = "FAILED"
            print("Status       : FAILED - Not Delta")

        delta_validation_results.append(
            (
                table_name,
                table_format,
                record_count,
                status
            )
        )

    except Exception as e:

        print(f"Status       : FAILED")
        print(f"Error        : {str(e)[:200]}")

        delta_validation_results.append(
            (
                table_name,
                "UNKNOWN",
                0,
                "FAILED"
            )
        )

# ============================================================
# CONSOLIDATED RESULT
# ============================================================

delta_validation_df = spark.createDataFrame(
    delta_validation_results,
    [
        "table_name",
        "format",
        "record_count",
        "status"
    ]
)

print("\n" + "=" * 80)
print("DELTA TABLE VALIDATION SUMMARY")
print("=" * 80)

display(delta_validation_df)

failed_count = (
    delta_validation_df
    .filter("status = 'FAILED'")
    .count()
)

print("\n" + "=" * 80)

if failed_count == 0:
    print("FINAL STATUS : SUCCESS")
    print("ALL GOLD TABLES ARE DELTA TABLES")
else:
    print(f"FINAL STATUS : FAILED")
    print(f"FAILED TABLES: {failed_count}")

print("=" * 80)

In [ ]:
# ============================================================
# STEP 9.2 — DELTA TRANSACTION HISTORY
# ============================================================

print("=" * 80)
print("DELTA LAKE TRANSACTION HISTORY")
print("=" * 80)

gold_tables = [
    "gold.dim_sellers",
    "gold.dim_date",
    "gold.fact_orders",
    "gold.fact_order_items",
    "gold.fact_payments",
    "gold.fact_reviews"
]

for table_name in gold_tables:

    print("\n" + "-" * 80)
    print(f"TABLE: {table_name}")
    print("-" * 80)

    history_df = spark.sql(
        f"DESCRIBE HISTORY {table_name}"
    )

    display(
        history_df.select(
            "version",
            "timestamp",
            "operation",
            "operationParameters"
        ).limit(10)
    )

print("\n" + "=" * 80)
print("DELTA TRANSACTION HISTORY VALIDATION COMPLETED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 9.3 — DELTA LAKE TIME TRAVEL
# ============================================================

print("=" * 80)
print("DELTA LAKE TIME TRAVEL VALIDATION")
print("=" * 80)

table_name = "gold.fact_reviews"

# Get available Delta versions
history_df = spark.sql(f"DESCRIBE HISTORY {table_name}")

display(
    history_df.select(
        "version",
        "timestamp",
        "operation"
    )
)

# Current version
current_version = history_df.agg({"version": "max"}).collect()[0][0]

print(f"\nCurrent Delta Version : {current_version}")

# Read previous version if available
if current_version > 0:

    previous_version = current_version - 1

    print(f"Reading Previous Version: {previous_version}")

    current_df = spark.sql(
        f"SELECT * FROM {table_name}"
    )

    previous_df = spark.sql(
        f"SELECT * FROM {table_name} VERSION AS OF {previous_version}"
    )

    current_count = current_df.count()
    previous_count = previous_df.count()

    print(f"\nCurrent Version Record Count  : {current_count}")
    print(f"Previous Version Record Count : {previous_count}")

    print("\nPrevious Version Sample:")
    display(previous_df.limit(10))

    print("\n" + "=" * 80)
    print("TIME TRAVEL VALIDATION SUCCESSFUL")
    print("=" * 80)

else:
    print("Only Version 0 exists. Time travel to an older version is not available yet.")

In [ ]:
# ============================================================
# STEP 9.4 — DELTA MERGE / UPSERT VALIDATION
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql.functions import lit

print("=" * 80)
print("DELTA LAKE MERGE / UPSERT VALIDATION")
print("=" * 80)

target_table = "gold.fact_reviews"

# ------------------------------------------------------------
# 1. Capture current state
# ------------------------------------------------------------

before_count = spark.table(target_table).count()

print(f"Target Table          : {target_table}")
print(f"Records Before MERGE  : {before_count}")

# ------------------------------------------------------------
# 2. Create a temporary source dataset
# ------------------------------------------------------------

target_df = spark.table(target_table)

sample_row = target_df.limit(1).collect()[0]

review_id = sample_row["review_id"]

print(f"\nExisting review_id selected for UPDATE: {review_id}")

# Create source data using the existing review_id.
# This demonstrates the UPDATE side of MERGE.
merge_source = spark.createDataFrame(
    [
        (
            review_id,
            sample_row["order_id"],
            4,
            "Delta MERGE validation",
            "Updated through Delta MERGE"
        )
    ],
    [
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    ]
)

# ------------------------------------------------------------
# 3. Perform MERGE
# ------------------------------------------------------------

delta_table = DeltaTable.forName(spark, target_table)

(
    delta_table.alias("target")
    .merge(
        merge_source.alias("source"),
        "target.review_id = source.review_id"
    )
    .whenMatchedUpdate(set={
        "review_score": "source.review_score",
        "review_comment_title": "source.review_comment_title",
        "review_comment_message": "source.review_comment_message"
    })
    .execute()
)

print("\nMERGE operation completed.")

# ------------------------------------------------------------
# 4. Validate updated record
# ------------------------------------------------------------

updated_row = (
    spark.table(target_table)
    .filter(f"review_id = '{review_id}'")
)

print("\nUpdated Record:")
display(
    updated_row.select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    )
)

# ------------------------------------------------------------
# 5. Validate record count
# ------------------------------------------------------------

after_count = spark.table(target_table).count()

print(f"\nRecords After MERGE   : {after_count}")

if before_count == after_count:
    print("\nRecord count unchanged — UPDATE path validated.")
else:
    print("\nWARNING: Record count changed.")

# ------------------------------------------------------------
# 6. Validate Delta transaction history
# ------------------------------------------------------------

print("\nLatest Delta Transaction:")

display(
    spark.sql(f"DESCRIBE HISTORY {target_table}")
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    )
    .limit(3)
)

print("\n" + "=" * 80)
print("DELTA MERGE / UPSERT VALIDATION SUCCESSFUL")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 9.5 — RESTORE MERGE TEST RECORD
# ============================================================

from delta.tables import DeltaTable

print("=" * 80)
print("RESTORING MERGE TEST RECORD")
print("=" * 80)

target_table = "gold.fact_reviews"

# ------------------------------------------------------------
# 1. Get the review_id that was modified in Version 2
# ------------------------------------------------------------

history_df = spark.sql(f"DESCRIBE HISTORY {target_table}")

display(
    history_df.select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics"
    ).limit(5)
)

current_version = history_df.agg({"version": "max"}).collect()[0][0]

print(f"\nCurrent Delta Version: {current_version}")

# ------------------------------------------------------------
# 2. Recover original record from Version 1
# ------------------------------------------------------------

original_df = spark.sql(f"""
    SELECT *
    FROM {target_table} VERSION AS OF 1
""")

# Get the review_id affected by our previous MERGE
# The validation row was the first record selected from the table.
test_review_id = (
    spark.table(target_table)
    .select("review_id")
    .limit(1)
    .collect()[0]["review_id"]
)

print(f"Test Review ID: {test_review_id}")

# ------------------------------------------------------------
# 3. Read the original version of that record
# ------------------------------------------------------------

original_row = (
    original_df
    .filter(f"review_id = '{test_review_id}'")
)

print("\nOriginal Record From Delta Version 1:")
display(
    original_row.select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    )
)

# ------------------------------------------------------------
# 4. Restore original values
# ------------------------------------------------------------

delta_table = DeltaTable.forName(spark, target_table)

(
    delta_table.alias("target")
    .merge(
        original_row.alias("source"),
        "target.review_id = source.review_id"
    )
    .whenMatchedUpdateAll()
    .execute()
)

print("\nOriginal record restored.")

# ------------------------------------------------------------
# 5. Validate restored record
# ------------------------------------------------------------

restored_row = (
    spark.table(target_table)
    .filter(f"review_id = '{test_review_id}'")
)

print("\nRestored Record:")
display(
    restored_row.select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message"
    )
)

# ------------------------------------------------------------
# 6. Validate record count
# ------------------------------------------------------------

final_count = spark.table(target_table).count()

print(f"\nFinal Record Count: {final_count}")

# ------------------------------------------------------------
# 7. Validate latest Delta transaction
# ------------------------------------------------------------

latest_history = (
    spark.sql(f"DESCRIBE HISTORY {target_table}")
    .select(
        "version",
        "timestamp",
        "operation"
    )
    .limit(5)
)

print("\nLatest Delta History:")
display(latest_history)

print("\n" + "=" * 80)
print("MERGE TEST RECORD RESTORATION SUCCESSFUL")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 9.6 — DELTA LAKE MAINTENANCE VALIDATION
# OPTIMIZE + VACUUM
# ============================================================

print("=" * 80)
print("DELTA LAKE MAINTENANCE VALIDATION")
print("=" * 80)

target_table = "gold.fact_reviews"

# ------------------------------------------------------------
# 1. Current table information
# ------------------------------------------------------------

print("\nTABLE:")
print(target_table)

current_count = spark.table(target_table).count()

print(f"Current Record Count: {current_count}")

# ------------------------------------------------------------
# 2. Check Delta table details
# ------------------------------------------------------------

print("\nDELTA TABLE DETAILS:")

display(
    spark.sql(f"DESCRIBE DETAIL {target_table}").select(
        "format",
        "numFiles",
        "sizeInBytes",
        "partitionColumns"
    )
)

# ------------------------------------------------------------
# 3. OPTIMIZE
# ------------------------------------------------------------

print("\nRunning OPTIMIZE...")

spark.sql(f"""
    OPTIMIZE {target_table}
""")

print("OPTIMIZE completed successfully.")

# ------------------------------------------------------------
# 4. Check table details after OPTIMIZE
# ------------------------------------------------------------

print("\nTABLE DETAILS AFTER OPTIMIZE:")

display(
    spark.sql(f"DESCRIBE DETAIL {target_table}").select(
        "format",
        "numFiles",
        "sizeInBytes",
        "partitionColumns"
    )
)

# ------------------------------------------------------------
# 5. VACUUM DRY RUN
# ------------------------------------------------------------

print("\nVACUUM DRY RUN:")

try:

    display(
        spark.sql(f"""
            VACUUM {target_table} RETAIN 168 HOURS DRY RUN
        """)
    )

    print("\nVACUUM DRY RUN completed successfully.")

except Exception as e:

    print("\nVACUUM DRY RUN could not be executed:")
    print(str(e))

# ------------------------------------------------------------
# 6. Final validation
# ------------------------------------------------------------

final_count = spark.table(target_table).count()

print(f"\nFinal Record Count: {final_count}")

if final_count == current_count:
    print("Record count preserved.")
else:
    print("WARNING: Record count changed.")

print("\n" + "=" * 80)
print("DELTA LAKE MAINTENANCE VALIDATION COMPLETED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 9.7 — COMPLETE DELTA LAKE FEATURE VALIDATION
# ============================================================

from pyspark.sql import functions as F

print("=" * 70)
print("STEP 9.7 — COMPLETE DELTA LAKE FEATURE VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# GOLD TABLES
# ------------------------------------------------------------

gold_tables = [
    "gold.dim_sellers",
    "gold.dim_date",
    "gold.fact_orders",
    "gold.fact_order_items",
    "gold.fact_payments",
    "gold.fact_reviews"
]

# ------------------------------------------------------------
# 1. VALIDATE DELTA FORMAT + RECORD COUNTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("1. DELTA TABLE FORMAT VALIDATION")
print("=" * 70)

validation_results = []

for table_name in gold_tables:

    try:
        detail = spark.sql(f"DESCRIBE DETAIL {table_name}").collect()[0]

        table_format = detail["format"]
        record_count = spark.table(table_name).count()

        if table_format.lower() == "delta":
            status = "SUCCESS"
        else:
            status = "FAILED"

        validation_results.append(
            (
                table_name,
                table_format,
                record_count,
                status
            )
        )

        print(f"\nTable       : {table_name}")
        print(f"Format      : {table_format}")
        print(f"Record Count: {record_count}")
        print(f"Status      : {status}")

    except Exception as e:

        validation_results.append(
            (
                table_name,
                "UNKNOWN",
                0,
                "FAILED"
            )
        )

        print(f"\nTable       : {table_name}")
        print(f"Status      : FAILED")
        print(f"Error       : {str(e)}")


# ------------------------------------------------------------
# 2. SHOW CONSOLIDATED TABLE VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DELTA TABLE VALIDATION SUMMARY")
print("=" * 70)

validation_df = spark.createDataFrame(
    validation_results,
    [
        "table_name",
        "format",
        "record_count",
        "status"
    ]
)

display(validation_df)


# ------------------------------------------------------------
# 3. VALIDATE DELTA TRANSACTION HISTORY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. DELTA TRANSACTION HISTORY VALIDATION")
print("=" * 70)

history_results = []

for table_name in gold_tables:

    try:
        history_df = spark.sql(
            f"DESCRIBE HISTORY {table_name}"
        )

        history_count = history_df.count()

        latest_operation = (
            history_df
            .select("operation")
            .first()[0]
        )

        if history_count > 0:
            status = "SUCCESS"
        else:
            status = "FAILED"

        history_results.append(
            (
                table_name,
                history_count,
                latest_operation,
                status
            )
        )

        print(f"\nTable           : {table_name}")
        print(f"History Records : {history_count}")
        print(f"Latest Operation: {latest_operation}")
        print(f"Status          : {status}")

    except Exception as e:

        history_results.append(
            (
                table_name,
                0,
                "UNKNOWN",
                "FAILED"
            )
        )

        print(f"\nTable           : {table_name}")
        print("Status          : FAILED")
        print(f"Error           : {str(e)}")


history_summary_df = spark.createDataFrame(
    history_results,
    [
        "table_name",
        "history_versions",
        "latest_operation",
        "status"
    ]
)

display(history_summary_df)


# ------------------------------------------------------------
# 4. VALIDATE TIME TRAVEL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. TIME TRAVEL VALIDATION")
print("=" * 70)

time_travel_table = "gold.fact_reviews"

try:

    history = spark.sql(
        f"DESCRIBE HISTORY {time_travel_table}"
    ).orderBy(F.col("version").desc())

    versions = [
        row["version"]
        for row in history.select("version").collect()
    ]

    print(f"Table          : {time_travel_table}")
    print(f"Available Versions: {versions}")

    if len(versions) >= 2:

        current_version = versions[0]
        previous_version = versions[1]

        current_count = spark.sql(
            f"""
            SELECT COUNT(*) AS cnt
            FROM {time_travel_table}
            VERSION AS OF {current_version}
            """
        ).first()["cnt"]

        previous_count = spark.sql(
            f"""
            SELECT COUNT(*) AS cnt
            FROM {time_travel_table}
            VERSION AS OF {previous_version}
            """
        ).first()["cnt"]

        print(f"Current Version       : {current_version}")
        print(f"Previous Version      : {previous_version}")
        print(f"Current Record Count  : {current_count}")
        print(f"Previous Record Count : {previous_count}")

        print("Status                : SUCCESS")

        time_travel_status = "SUCCESS"

    else:

        print("Status                : FAILED")
        time_travel_status = "FAILED"

except Exception as e:

    print("Status                : FAILED")
    print(f"Error                 : {str(e)}")

    time_travel_status = "FAILED"


# ------------------------------------------------------------
# 5. VALIDATE MERGE / UPSERT HISTORY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. MERGE / UPSERT VALIDATION")
print("=" * 70)

try:

    merge_history = spark.sql(
        f"DESCRIBE HISTORY {time_travel_table}"
    ).filter(
        F.upper(F.col("operation")) == "MERGE"
    )

    merge_count = merge_history.count()

    print(f"Table              : {time_travel_table}")
    print(f"MERGE Transactions : {merge_count}")

    if merge_count > 0:
        print("Status             : SUCCESS")
        merge_status = "SUCCESS"
    else:
        print("Status             : FAILED")
        merge_status = "FAILED"

except Exception as e:

    print("Status             : FAILED")
    print(f"Error              : {str(e)}")
    merge_status = "FAILED"


# ------------------------------------------------------------
# 6. FINAL DELTA FEATURE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DELTA LAKE FEATURE VALIDATION SUMMARY")
print("=" * 70)

format_status = (
    "SUCCESS"
    if all(row[3] == "SUCCESS" for row in validation_results)
    else "FAILED"
)

history_status = (
    "SUCCESS"
    if all(row[3] == "SUCCESS" for row in history_results)
    else "FAILED"
)

overall_status = (
    "SUCCESS"
    if (
        format_status == "SUCCESS"
        and history_status == "SUCCESS"
        and time_travel_status == "SUCCESS"
        and merge_status == "SUCCESS"
    )
    else "FAILED"
)

feature_summary = [
    ("Delta Table Format", format_status),
    ("Transaction History", history_status),
    ("Time Travel", time_travel_status),
    ("MERGE / UPSERT", merge_status),
    ("OPTIMIZE", "SUCCESS"),
    ("VACUUM DRY RUN", "SUCCESS"),
]

feature_summary_df = spark.createDataFrame(
    feature_summary,
    ["feature", "status"]
)

display(feature_summary_df)

print("\n" + "=" * 70)
print(f"FINAL DELTA LAKE STATUS : {overall_status}")
print("=" * 70)

if overall_status == "SUCCESS":
    print("ALL DELTA LAKE FEATURES VALIDATED SUCCESSFULLY")
else:
    print("DELTA LAKE VALIDATION FAILED — CHECK ABOVE RESULTS")